# SEC Filings Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kineviz/fortune500/blob/main/pipeline.ipynb)

Full pipeline from scraping 10-K filings through BigQuery AI extraction and property graph creation.

## Pipeline Steps
1. **Google Cloud authentication** – gcloud auth
2. **Scraper** – Download 10-K filings from SEC.gov
3. **Parser** – Parse SGML to Markdown (optional, for viewing)
4. **Extract Sections** – Parse SGML to JSONL (Item 1, 1A, 7; feeds BigQuery)
5. **Upload JSONL to GCS** – Sync JSON to BigQuery load bucket
6. **Init Tables** – Create sections, insights, staging tables
7. **Extraction** – AI extraction (Gemini) → insights
8. **Create Graph** – Transform insights to graph_edges
9. **Prepare Property Graph** – Node and edge tables
10. **Add Action to Market** – market_action column
11. **Create Property Graph DDL** – SecGraph property graph

### 0.0 GCP Environment Setup Checklist
Before running this notebook, ensure your Google Cloud environment is fully configured:

1.  **API Enablement**: Enable [BigQuery API](https://console.cloud.google.com/marketplace/product/google/bigquery.googleapis.com) and [Vertex AI API](https://console.cloud.google.com/marketplace/product/google/aiplatform.googleapis.com).
2.  **Billing**: Ensure your project is linked to an active **[Billing Account](https://console.cloud.google.com/billing/enable)**.
3.  **BigQuery AI Connection**:
    - Create a Cloud Resource Connection named **`vertex_ai_connection`** in the **US** location.
    - Grant the connection's Service Account the **`roles/aiplatform.user`** role.
    - **Run the cell below** to automate this setup.

## Configuration

Set the environment constants below. These variables control GCP authentication, local and cloud storage, and pipeline depth.

### GCP Settings
- **`GCP_PROJECT`**: Your Google Cloud Project ID. Find this in the [GCP Console Project Dashboard](https://console.cloud.google.com/home/dashboard). Ensure you have the [BigQuery API](https://console.cloud.google.com/marketplace/product/google/bigquery.googleapis.com) and [Vertex AI API](https://console.cloud.google.com/marketplace/product/google/aiplatform.googleapis.com) enabled.
- **`GCS_BUCKET`**: The Google Cloud Storage bucket used for staging data for BigQuery load. Format: `gs://your-bucket-name`.
- **`BQ_LOCATION`**: The regional location for your BigQuery datasets (e.g., `US`, `EU`).
- **`BQ_DATASET`**: The name of the BigQuery Dataset (default: `sec_filings`). Allows isolating runs.
- **`GEMINI_MODEL`**: The Gemini model version used for text extraction (e.g., `gemini-3.5-pro`).
- **`BQ_MODEL`**: The database identifier name assigned to the LLM model connection (default: `gemini_pro_latest`).
- **Billing**: BigQuery AI functions require an active billing account. Ensure it is enabled [here](https://console.cloud.google.com/billing/enable).

### Local Directories
These variables define where data is stored on your local machine or external drive.
- **`SGML_DIR`**: Root directory for raw `.sgml` files from SEC.gov.
- **`MARKDOWN_DIR`**: Directory for converted `.md` files.
- **`JSON_DIR`**: Directory for sectioned JSONL files (Item 1, 1A, 7).

### Scraper Settings
- **`SCRAPER_LIMIT`**: The number of companies to pull from the companies list (e.g., 5, 20, 500). Use a small number to test.
- **`SCRAPER_LAST_N_YEARS`**: The number of historical years to scrape for each company.

In [ ]:
# Configure these for your environment
import os

GCP_PROJECT = "YOUR_PROJECT_ID"  # Change to your project ID
GCS_BUCKET = "gs://YOUR_BUCKET"  # Change to your bucket
BQ_LOCATION = "US"
BQ_DATASET = "fortune500"  # Change to map to a different BigQuery dataset
GEMINI_MODEL = "gemini-3.1-pro-preview"  # Updated to a stable model version
BQ_MODEL = "gemini_pro_3"  # Name of the BigQuery model object

# Local data directories
SGML_DIR = "./data/sgml"
MARKDOWN_DIR = "./data/markdown"
JSON_DIR = "./data/json"

# Pipeline depth
SCRAPER_LIMIT = 2           # Companies from list.csv (use a small number for testing, the max is 500)
SCRAPER_LAST_N_YEARS = 2      # Years of filings to fetch
SCRAPER_OUTPUT = SGML_DIR

# Derived project constants
BQ_PROJECT = GCP_PROJECT

# Assert configuration is updated
assert GCP_PROJECT != "YOUR_PROJECT_ID", "Update GCP_PROJECT at the top of the notebook."
assert GCS_BUCKET != "gs://YOUR_BUCKET", "Update GCS_BUCKET at the top of the notebook."
assert "-" not in BQ_DATASET, "BQ_DATASET cannot contain '-' (use underscores instead)."

# Create local data directories if they don't exist
for d in [SGML_DIR, MARKDOWN_DIR, JSON_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"Verified directories: {SGML_DIR}, {MARKDOWN_DIR}, {JSON_DIR}")

## 0. Colab Setup
If you are running this in Google Colab, you need to clone the repository and install requirements to access the custom scraper scripts and SQL files.

In [ ]:
import sys
import os
from IPython import get_ipython
ipy = get_ipython()
if 'google.colab' in sys.modules:
    if not os.path.exists('01_scraper.py'):
        ipy.system('git clone https://github.com/Kineviz/fortune500.git')
        os.chdir('fortune500')
ipy.system('pip install -r requirements.txt')


## 1. Setup & Google Cloud Authentication

In [ ]:
# Using GCP_PROJECT from configuration above

if GCP_PROJECT != "YOUR_PROJECT_ID":
    from IPython import get_ipython
    import subprocess
    import google.auth
    ipy = get_ipython()
    ipy.system(f"gcloud config set project {GCP_PROJECT}")

    # In Colab, this sets up Application Default Credentials for Python clients
    if 'google.colab' in sys.modules:
        from google.colab import auth as colab_auth
        colab_auth.authenticate_user()
    else:
        ipy.system("gcloud auth login")
        ipy.system("gcloud auth application-default login")

    result = subprocess.run(["gcloud", "config", "get-value", "project"], capture_output=True, text=True)
    current = (result.stdout or "").strip() or "(unset)"
    print(f"✓ Project verified: {current}") if current == GCP_PROJECT else print(f"⚠ Current: {current}")

    # Verify ADC is available to BigQuery Python client
    creds, adc_project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
    print(f"✓ ADC ready (quota project: {getattr(creds, 'quota_project_id', None) or adc_project or GCP_PROJECT})")
else:
    raise ValueError("Set GCP_PROJECT in the Configuration cell at the beginning of the notebook.")

In [ ]:
import subprocess
import os
import sys
import json

def check_billing(project_id):
    print(f"Checking billing status for {project_id}...")
    try:
        # Using gcloud to verify if billing is enabled on the given project
        result = subprocess.run(
            ["gcloud", "billing", "projects", "describe", project_id, "--format=json"],
            capture_output=True, text=True, check=True
        )
        billing_info = json.loads(result.stdout)
        if billing_info.get("billingEnabled"):
            print("\u2705 Billing is enabled! You are ready to use BigQuery AI.")
        else:
            print("\u274c ERROR: Billing is NOT enabled for this project.")
            print("BigQuery AI will not function until billing is enabled in the Google Cloud Console.")
            print(f"Visit: https://console.cloud.google.com/billing/enable?project={project_id}")
            raise Exception("Billing not enabled")
    except subprocess.CalledProcessError as e:
        print("\u26a0 Could not verify billing status automatically. Output:")
        print(e.stderr)
        print("\nEnsure you are authenticated and have permissions (roles/billing.projectManager).")

if GCP_PROJECT and GCP_PROJECT != "YOUR_PROJECT_ID":
    check_billing(GCP_PROJECT)


### Ensure we're in the project root (where 01_scraper.py, SQL files, list.csv live)


In [ ]:
import os

def find_project_root():
    cwd = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(cwd, "01_scraper.py")):
            return cwd
        parent = os.path.dirname(cwd)
        if parent == cwd:
            break
        cwd = parent
    return os.getcwd()

root = find_project_root()
os.chdir(root)
print(f"Working directory: {os.getcwd()}")

### Verify GCS bucket exists and is accessible


In [ ]:
import subprocess
import os
import sys

result = subprocess.run(
    ["gsutil", "ls", GCS_BUCKET],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print(f"✓ Bucket accessible: {GCS_BUCKET}")
else:
    print(f"Bucket not found. Creating {GCS_BUCKET}...")
    create = subprocess.run(["gsutil", "mb", "-l", BQ_LOCATION, GCS_BUCKET], capture_output=True, text=True)
    if create.returncode == 0:
        print(f"✓ Bucket created: {GCS_BUCKET}")
    else:
        print(f"⚠ Failed to create bucket: {create.stderr or create.stdout}")

In [ ]:
import asyncio
import sys

# Add project root to path
sys.path.insert(0, os.getcwd())

from importlib.util import spec_from_file_location, module_from_spec

# Load scraper module
spec = spec_from_file_location("scraper", "01_scraper.py")
scraper_mod = module_from_spec(spec)
spec.loader.exec_module(scraper_mod)

## 2. Run Scraper (01_scraper.py)

In [ ]:
scraper = scraper_mod.SECScraper(
    limit=SCRAPER_LIMIT,
    last_n_years=SCRAPER_LAST_N_YEARS,
    workers=5,
    output_dir=SCRAPER_OUTPUT,
    dry_run=False,
)

# Scraper has internal tqdm; get expected count for context
import pandas as pd
try:
    n_companies = min(SCRAPER_LIMIT, len(pd.read_csv("list.csv")))
    print(f"Scraping up to {n_companies} companies ({SCRAPER_LAST_N_YEARS} years each)...")
except FileNotFoundError:
    n_companies = SCRAPER_LIMIT
await scraper.run()  # Jupyter has its own event loop; use await instead of asyncio.run()

In [ ]:
import subprocess
import os
import sys
from tqdm.auto import tqdm

## 3. Run Parser (02_parser.py) (Optional)

In [ ]:
# Count SGML filings to process
n_sgml = sum(1 for r, d, f in os.walk(SGML_DIR) if "full-submission.txt" in f)
with tqdm(total=max(1, n_sgml), desc="Parsing SGML to Markdown", unit="filing") as pbar:
    subprocess.run(
        [sys.executable, "02_parser.py", "--input_base", SGML_DIR, "--output_base", MARKDOWN_DIR],
        check=True,
        cwd=os.getcwd(),
    )
    pbar.update(max(1, n_sgml))

## 4. Extract Sections (03_extract_sections.py)

In [ ]:
import subprocess
import os
import sys
from tqdm.auto import tqdm

# Count SGML filings to process
n_sgml = sum(1 for r, d, f in os.walk(SGML_DIR) if "full-submission.txt" in f)
with tqdm(total=max(1, n_sgml), desc="Extracting sections to JSONL", unit="section") as pbar:
    subprocess.run(
        [sys.executable, "03_extract_sections.py", "--input_base", SGML_DIR, "--output_base", JSON_DIR],
        check=True,
        cwd=os.getcwd(),
    )
    pbar.update(max(1, n_sgml))

## 5. Upload JSONL to GCS

In [ ]:
import os
import glob
from IPython import get_ipython
from tqdm.auto import tqdm
json_src = os.path.abspath(JSON_DIR)
os.makedirs(json_src, exist_ok=True)
json_files = glob.glob(os.path.join(json_src, "*", "*", "sections.jsonl"))
n_json = len(json_files)
if n_json == 0:
    print(f"⚠ {json_src} is empty. Run Steps 2–4 (scraper, parser, extract sections) first, then re-run this cell.")
else:
    with tqdm(total=n_json, desc="Uploading JSONL to GCS", unit="file") as pbar:
        get_ipython().system(f'gsutil -m rsync -r {json_src}/ {GCS_BUCKET}/json')
        pbar.update(n_json)
    print(f"✓ Synced {n_json} section files to {GCS_BUCKET}/json")

## 6. BigQuery Extraction & Python Normalization Pipeline

This hybrid pipeline uses BigQuery's native `AI.GENERATE_TEXT` to extract insights, then exports them to run the Python entity normalization locally.

### 6.1 Initialize Tables and Load Sections

Initialize the `sections` and `insights` tables in BigQuery, and load the raw sections from GCS.

In [ ]:
import subprocess
import google.auth
from google.cloud import bigquery

if 'google.colab' in __import__('sys').modules:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()

creds, adc_project = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
client = bigquery.Client(project=GCP_PROJECT, credentials=creds)
print(f"✓ BigQuery client ready (ADC project: {adc_project or GCP_PROJECT})")

def run_bq_query(filename):
    with open(filename, 'r') as f:
        sql = f.read()
    sql = sql.replace('sec_filings.', f"{BQ_DATASET}.")
    sql = sql.replace('sec_filings;', f"{BQ_DATASET};")
    print(f"Executing {filename}...")
    job = client.query(sql)
    job.result()
    print(f"✓ Executed {filename}")

run_bq_query("04_init_tables.sql")

print("Loading sections into BigQuery...")
sections_schema = "filing_id:STRING,company:STRING,company_name:STRING,cik:STRING,sic:STRING,irs_number:STRING,state_of_inc:STRING,org_name:STRING,sec_file_number:STRING,film_number:STRING,business_street_1:STRING,business_street_2:STRING,business_city:STRING,business_state:STRING,business_zip:STRING,business_phone:STRING,mail_street_1:STRING,mail_street_2:STRING,mail_city:STRING,mail_state:STRING,mail_zip:STRING,filing_url:STRING,year:INTEGER,section_id:STRING,content:STRING"
sections_uri = f"{GCS_BUCKET}/json/*/*/sections.jsonl"

type_map = {"STRING": "STRING", "INTEGER": "INT64"}
schema = []
for col in sections_schema.split(','):
    name, typ = col.split(':')
    schema.append(bigquery.SchemaField(name, type_map.get(typ, typ)))

table_id = f"{GCP_PROJECT}.{BQ_DATASET}.sections"
load_config = bigquery.LoadJobConfig(
    schema=schema,
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

load_job = client.load_table_from_uri(sections_uri, table_id, job_config=load_config)
load_job.result()
print("✓ Sections loaded")


### 6.2 Run AI Extraction in BigQuery

Extract insights using `AI.GENERATE_TEXT`.

In [ ]:
run_bq_query("05_extraction.sql")

### 6.3 Export Insights and Run Python Normalization

Export the `insights` table to GCS, download it locally, and run the Python transformation and normalization scripts.

In [ ]:
import os

print("Exporting insights to GCS...")
subprocess.run(["bq", "extract", "--destination_format=NEWLINE_DELIMITED_JSON", f"{GCP_PROJECT}:{BQ_DATASET}.insights", f"{GCS_BUCKET}/parquets/{GEMINI_MODEL}/insights.jsonl"], check=True)

print("Downloading insights locally...")
local_insights_dir = f"python_pipeline/output/{GEMINI_MODEL}/extractions"
os.makedirs(local_insights_dir, exist_ok=True)
subprocess.run(["gsutil", "cp", f"{GCS_BUCKET}/parquets/{GEMINI_MODEL}/insights.jsonl", f"{local_insights_dir}/insights.jsonl"], check=True)

print("Running Python Parquet Pipeline (Transformation & Normalization)...")
!cd python_pipeline && uv run python transform.py
!cd python_pipeline && uv run python entity_normalization/resolve_competitors.py
!cd python_pipeline && uv run python entity_normalization/categorize_risks.py
!cd python_pipeline && uv run python entity_normalization/categorize_markets.py
!cd python_pipeline && uv run python entity_normalization/categorize_competitor_markets.py

## 7. Load Parquet files to BigQuery

Upload the generated Parquet files to Google Cloud Storage and load them into native BigQuery tables.

In [ ]:
PARQUET_LOCAL_DIR = f"python_pipeline/output/{GEMINI_MODEL}/parquet"
gcs_parquet_path = f"{GCS_BUCKET}/parquets/{GEMINI_MODEL}/parquet"
print(f"Uploading {PARQUET_LOCAL_DIR} to {gcs_parquet_path} ...")

# Use gsutil rsync to copy the folder up to GCS
rsync_cmd = ["gsutil", "-m", "rsync", "-r", PARQUET_LOCAL_DIR, gcs_parquet_path]
subprocess.run(rsync_cmd, check=True)
print("Upload complete!")

In [ ]:
tables_to_load = {
    # -- Nodes (Base) --
    "nodes_company": "nodes_company.parquet",
    "nodes_document": "nodes_document.parquet",
    "nodes_section": "nodes_section.parquet",
    "nodes_reference": "nodes_reference.parquet",
    "nodes_opportunity": "nodes_opportunity.parquet",
    "nodes_competitor": "nodes_competitor.parquet",
    # -- Nodes (Categorized / Normalized Replacements) --
    "nodes_market": "nodes_market_categorized.parquet",
    "nodes_risk": "nodes_risk_categorized.parquet",
    # -- Nodes (Normalization Taxonomy) --
    "nodes_normalized_competitor": "nodes_normalized_competitor.parquet",
    "nodes_geographic_region": "nodes_geographic_region.parquet",
    "nodes_market_category": "nodes_market_category.parquet",
    "nodes_risk_category": "nodes_risk_category.parquet",
    
    # -- Edges (Base) --
    "edges_filed": "edges_filed.parquet",
    "edges_doc_contains_section": "edges_doc_contains_section.parquet",
    "edges_section_contains_ref": "edges_section_contains_ref.parquet",
    "edges_entering": "edges_entering.parquet",
    "edges_exiting": "edges_exiting.parquet",
    "edges_expanding": "edges_expanding.parquet",
    "edges_faces_risk": "edges_faces_risk.parquet",
    "edges_pursuing": "edges_pursuing.parquet",
    "edges_competes": "edges_competes.parquet",
    "edges_market_has_reference": "edges_market_has_reference.parquet",
    "edges_risk_has_reference": "edges_risk_has_reference.parquet",
    "edges_opportunity_has_reference": "edges_opportunity_has_reference.parquet",
    "edges_competitor_has_reference": "edges_competitor_has_reference.parquet",
    
    # -- Edges (Normalization Links) --
    "edges_instance_of": "edges_instance_of.parquet",
    "edges_subsidiary_of": "edges_subsidiary_of.parquet",
    "edges_has_risk_category": "edges_has_risk_category.parquet",
    "edges_in_region": "edges_in_region.parquet",
    "edges_in_product_category": "edges_in_product_category.parquet",
    "edges_in_market_category": "edges_in_market_category.parquet"
}

for bq_table_name, parquet_file in tables_to_load.items():
    uri = f"{gcs_parquet_path}/{parquet_file}"
    query = f"""
    LOAD DATA OVERWRITE `{GCP_PROJECT}.{BQ_DATASET}.{bq_table_name}`
    FROM FILES (
      format = 'PARQUET',
      uris = ['{uri}']
    );
    """
    print(f"Loading {bq_table_name} from {parquet_file}...")
    job = client.query(query)
    job.result()  # Waits for the query to finish
print("\nAll Parquet tables successfully loaded into BigQuery!")

## 8. Re-create the Property Graph DDL

Update the `SecGraph` Property Graph to include the new taxonomy nodes and their connective edges.

In [ ]:
run_bq_query("06_create_property_graph_ddl.sql")
print("\nPipeline completed successfully!")